# BFS LTX-2.5 R128 Face-Swap — Kaggle Gradio + Cloudflare

**Standalone notebook.** This does not reuse the LTX video-generation notebook's Wan2GP engine. It runs the creator's LTX-2.5 BFS workflow through a **headless ComfyUI backend** (no ComfyUI browser UI), with a custom **Gradio front end** and a **Cloudflare Quick Tunnel**.

- Target runtime: Kaggle GPU T4 x2, approximately 30 GB system RAM.
- BFS adapter: `head_swap_ltx25_r128_v1.1.safetensors` (kept in original precision).
- Base transformer: official LTX-2.5 INT8 ConvRot checkpoint from the creator's workflow.
- No extra quantization of the BFS LoRA. The official base checkpoint is already INT8 ConvRot.
- The workflow JSON and required assets are downloaded from their official repositories.

**Important hardware note:** Two T4 cards do not automatically become one 30 GB GPU. This notebook exposes both cards to CUDA, but ComfyUI's default model placement is not guaranteed to shard a single transformer across both cards. The backend uses low-VRAM/offload mode; it may leave GPU 1 mostly idle depending on the installed ComfyUI/LTX loader. A true split requires explicit model-parallel support in the workflow's loader. Do not assume 30 GB pooled VRAM.

**Access:** Only use source/reference images and videos you own or have explicit permission to edit. Respect the LTX-2 Community License and Hugging Face gated-model access.

Run cells in order. The first model download is large and requires accepting the LTX-2.5 model terms on Hugging Face and setting `HF_TOKEN` in Kaggle Secrets (or entering it below).

In [1]:
#@title 1. Runtime check + configuration
import os, sys, json, time, shutil, subprocess, pathlib, textwrap, socket, requests, psutil
from concurrent.futures import ThreadPoolExecutor

assert os.path.isdir("/kaggle"), "This notebook is designed for Kaggle."
ROOT = "/kaggle/working/BFS_LTX25"
COMFY_DIR = f"{ROOT}/ComfyUI"
CUSTOM_NODES = f"{COMFY_DIR}/custom_nodes/ComfyUI-BFSNodes"
MODELS = f"/kaggle/tmp/bfs_ltx25_models"
INPUT_DIR = f"{COMFY_DIR}/input"
OUTPUT_DIR = f"{COMFY_DIR}/output"
WORKFLOW_REPO = "Alissonerdx/BFS-Best-Face-Swap-Video"
WORKFLOW_FILE = f"{ROOT}/workflow_ltx2.5_v1.1_head_swap_drag_and_drop.json"
WORKFLOW_API = f"{ROOT}/workflow_ltx2.5_v1.1_api.json"
COMFY_URL = "http://127.0.0.1:8188"
GRADIO_PORT = 7860

os.makedirs(ROOT, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

# HF_TOKEN can be stored in Kaggle -> Add-ons -> Secrets as HF_TOKEN.
HF_TOKEN = os.environ.get("HF_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = HF_TOKEN or UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

print("Python:", sys.version.split()[0])
print("RAM total/available:", round(psutil.virtual_memory().total/2**30,1),
      "/", round(psutil.virtual_memory().available/2**30,1), "GiB")
try:
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free",
         "--format=csv,noheader"], text=True))
except Exception as e:
    raise RuntimeError("Enable GPU T4 x2 in Kaggle Settings > Accelerator.") from e

import torch
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda,
      "| visible GPUs:", torch.cuda.device_count())
if torch.cuda.device_count() < 1:
    raise RuntimeError("No CUDA GPU found.")
if torch.cuda.device_count() < 2:
    print("WARNING: only one GPU is visible; T4 x2 is recommended.")

# Keep both GPUs visible; ComfyUI may still use one primary device unless its loader supports sharding.
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1" if torch.cuda.device_count() >= 2 else "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"

free_gb = shutil.disk_usage(MODELS).free / 2**30
print(f"Model cache: {MODELS} | free disk: {free_gb:.1f} GiB")
if free_gb < 38:
    print("WARNING: less than ~38 GiB free. The official BFS workflow needs several large checkpoints.")


Python: 3.13.16
RAM total/available: 31.3 / 29.9 GiB
0, Tesla T4, 15360 MiB, 14912 MiB
1, Tesla T4, 15360 MiB, 14912 MiB

Torch: 2.11.0+cu130 | CUDA: 13.0 | visible GPUs: 2
Model cache: /kaggle/tmp/bfs_ltx25_models | free disk: 966.9 GiB


In [3]:

import os
import sys
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/working/BFS_LTX25")
COMFY = ROOT / "ComfyUI"
CUSTOM_NODES = COMFY / "custom_nodes"
BFS_NODES = CUSTOM_NODES / "ComfyUI-BFSNodes"

def run(cmd, cwd=None):
    print("\n$", " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=cwd, check=True)

# 1. Clone ComfyUI only if it does not already exist
if not COMFY.exists():
    run([
        "git", "clone", "--depth", "1",
        "https://github.com/comfyanonymous/ComfyUI.git",
        str(COMFY)
    ])
else:
    print("ComfyUI already exists — skipping clone.")

# 2. Install ComfyUI requirements
run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements.txt"
], cwd=str(COMFY))

# 3. Clone BFS custom nodes only if missing
CUSTOM_NODES.mkdir(parents=True, exist_ok=True)

if not BFS_NODES.exists():
    run([
        "git", "clone", "--depth", "1",
        "https://github.com/alisson-anjos/ComfyUI-BFSNodes.git",
        str(BFS_NODES)
    ])
else:
    print("BFSNodes already exists — skipping clone.")

# 4. Install BFSNodes and Gradio dependencies
run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.55,<5",
    "safetensors",
    "opencv-python-headless",
    "scenedetect",
    "librosa",
    "scipy",
    "gradio==5.29.0",
    "requests",
    "huggingface_hub[hf_xet]<1.0",
    "hf_xet>=1.5.2"
])

# 5. Install comfy-ui2api directly from GitHub
run([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/edusouzaxGV/comfy-ui2api.git"
])

# 6. Verify imports
import gradio
import transformers
import comfy_ui2api

print("\n========== INSTALLATION CHECK ==========")
print("Python:", sys.version)
print("Gradio:", gradio.__version__)
print("Transformers:", transformers.__version__)
print("comfy_ui2api:", getattr(comfy_ui2api, "__version__", "Installed"))
print("ComfyUI:", COMFY.exists())
print("BFSNodes:", BFS_NODES.exists())
print("\nAll installation steps completed.")


ComfyUI already exists — skipping clone.

$ /usr/bin/python3 -m pip install -q -r requirements.txt
BFSNodes already exists — skipping clone.

$ /usr/bin/python3 -m pip install -q transformers>=4.55,<5 safetensors opencv-python-headless scenedetect librosa scipy gradio==5.29.0 requests huggingface_hub[hf_xet]<1.0 hf_xet>=1.5.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sigstore-models 0.0.6 requires pydantic>=2.12, but you have pydantic 2.11.10 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.10.0 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.11.10 which is incompatible.
python-fasthtml 0.14.14 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
google-adk 2.7.1 requires pydantic<3,>=2.12, but you have pydantic 2.11.10 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.



$ /usr/bin/python3 -m pip install -q git+https://github.com/edusouzaxGV/comfy-ui2api.git

========== INSTALLATION CHECK ==========
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Gradio: 5.29.0
Transformers: 4.57.6
comfy_ui2api: 0.1.0
ComfyUI: True
BFSNodes: True

All installation steps completed.


In [11]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Try all possible secret names
HF_TOKEN = None

for secret_name in ["HF_TOKEN", "HF key"]:
    try:
        value = user_secrets.get_secret(secret_name)
        if value and value.strip():
            HF_TOKEN = value.strip()
            print(f"✅ Hugging Face token loaded from Kaggle Secret: {secret_name}")
            break
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "HF token nahi mila. Kaggle Secrets mein "
        "'HF_TOKEN' ya 'HF key' naam se token add karo."
    )

# Make token available to Hugging Face libraries
import os
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("✅ HF_TOKEN configured successfully!")

✅ Hugging Face token loaded from Kaggle Secret: HF_TOKEN
✅ HF_TOKEN configured successfully!


In [16]:

# ============================================================
# CELL 3 — BFS LTX-2.5 MODEL DOWNLOAD + VERIFICATION
# Workspace: /kaggle/temp/BFS_LTX25
# ============================================================

import os
import sys
import shutil
import warnings
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

ROOT = "/kaggle/temp/BFS_LTX25"
COMFY_DIR = os.path.join(ROOT, "ComfyUI")

WORKFLOW_REPO = "Alissonerdx/BFS-Best-Face-Swap-Video"
WORKFLOW_FILE = "workflows/workflow_ltx2.5_v1.1_head_swap_drag_and_drop.json"
WORKFLOW_PATH = os.path.join(ROOT, WORKFLOW_FILE)

HF_HOME = "/kaggle/temp/huggingface"

os.makedirs(ROOT, exist_ok=True)
os.makedirs(COMFY_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)

os.environ["HF_HOME"] = HF_HOME
os.environ["HF_HUB_CACHE"] = os.path.join(HF_HOME, "hub")

# Force Hugging Face progress bars ON
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"

# ------------------------------------------------------------
# 2. HUGGING FACE AUTHENTICATION
# ------------------------------------------------------------

from huggingface_hub import (
    hf_hub_download,
    login,
    HfApi
)
from huggingface_hub.utils import enable_progress_bars

enable_progress_bars()

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()

        for secret_name in ["HF key", "HF_TOKEN"]:
            try:
                HF_TOKEN = secrets.get_secret(secret_name)
                if HF_TOKEN:
                    break
            except Exception:
                continue

    except Exception as e:
        print("Kaggle Secrets access warning:", str(e))

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN not found. Add your Hugging Face token "
        "in Kaggle Secrets as 'HF key' or 'HF_TOKEN'."
    )

login(
    token=HF_TOKEN,
    add_to_git_credential=False
)

try:
    user_info = HfApi().whoami(token=HF_TOKEN)
    print("HF authentication: OK")
    print("Authenticated account:", user_info.get("name", "Verified"))
except Exception as e:
    raise RuntimeError(
        "Hugging Face authentication failed. Check your token."
    ) from e

# ------------------------------------------------------------
# 3. COMFYUI MODEL DIRECTORIES
# ------------------------------------------------------------

MODEL_ROOT = os.path.join(COMFY_DIR, "models")

MODEL_DIRS = [
    "diffusion_models",
    "text_encoders",
    "vae",
    "latent_upscale_models",
    "loras"
]

for folder in MODEL_DIRS:
    os.makedirs(
        os.path.join(MODEL_ROOT, folder),
        exist_ok=True
    )

# ------------------------------------------------------------
# 4. DOWNLOAD BFS WORKFLOW JSON
# ------------------------------------------------------------

print("\nDownloading BFS workflow...")

os.makedirs(
    os.path.dirname(WORKFLOW_PATH),
    exist_ok=True
)

if (
    os.path.exists(WORKFLOW_PATH)
    and os.path.getsize(WORKFLOW_PATH) > 1000
):
    print("Workflow already exists. Skipping.")
else:
    workflow_download = hf_hub_download(
        repo_id=WORKFLOW_REPO,
        filename=WORKFLOW_FILE,
        token=HF_TOKEN,
        local_dir=ROOT,
        force_download=False
    )

    if os.path.abspath(workflow_download) != os.path.abspath(WORKFLOW_PATH):
        shutil.copy2(workflow_download, WORKFLOW_PATH)

    print("Workflow downloaded successfully.")

print("Workflow:", WORKFLOW_PATH)

# ------------------------------------------------------------
# 5. MODEL DOWNLOAD LIST
#
# Format:
# (repo_id, filename, comfyui_folder, required)
#
# required=False:
# Download failure will NOT crash the cell.
# ------------------------------------------------------------

MODEL_FILES = [

    # 1. LTX-2.5 Main Transformer
    (
        "Lightricks/LTX-2.5",
        "ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors",
        "diffusion_models",
        True
    ),

    # 2. Gemma 4 12B Text Encoder
    (
        "Lightricks/LTX-2.5",
        "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors",
        "text_encoders",
        True
    ),

    # 3. Gemma 4 E2B Prompt Enhancer
    (
        "Comfy-Org/gemma-4",
        "gemma4_e2b_it_bf16.safetensors",
        "text_encoders",
        False
    ),

    # 4. LTX Video VAE
    (
        "Lightricks/LTX-2.5",
        "ltx-2.5-video-vae-bf16.safetensors",
        "vae",
        True
    ),

    # 5. LTX Audio VAE
    (
        "Lightricks/LTX-2.5",
        "ltx-2.5-audio-vae-bf16.safetensors",
        "vae",
        True
    ),

    # 6. Latent Spatial Upscaler
    (
        "Lightricks/LTX-2.5",
        "ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors",
        "latent_upscale_models",
        True
    ),

    # 7. BFS Head Swap R128 LoRA
    (
        "Alissonerdx/BFS-Best-Face-Swap-Video",
        "ltx-2.5/head_swap_ltx25_r128_v1.1.safetensors",
        "loras",
        True
    ),

    # 8. Pixel Spatial Upscaler IC-LoRA
    # OPTIONAL: HF repo is gated; 403 will not crash this cell.
    (
        "Lightricks/LTX-2.5-22b-IC-LoRA-Pixel-Spatial-Upscaler",
        "ltx-2.5-22b-ic-lora-pixel-spatial-upscaler-x2-1.0.safetensors",
        "loras",
        False
    ),

    # 9. Qwen3-VL 4B Heretic INT8
    # Optional auto-prompt model
    (
        "ethanfel/Qwen3-VL-4B-Heretic-ComfyUI-ConvRot-INT8",
        "qwen3-vl-4b-heretic_int8_dynamic_convrot.safetensors",
        "text_encoders",
        False
    )
]

# ------------------------------------------------------------
# 6. HELPER FUNCTIONS
# ------------------------------------------------------------

def get_size_gb(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / (1024 ** 3)
    return 0.0


def download_one(item):

    repo, filename, folder, required = item

    name = os.path.basename(filename)

    target_dir = os.path.join(MODEL_ROOT, folder)
    target_path = os.path.join(target_dir, name)

    staged_path = os.path.join(ROOT, filename)

    os.makedirs(target_dir, exist_ok=True)

    # Already downloaded
    if (
        os.path.isfile(target_path)
        and os.path.getsize(target_path) > 1024 * 1024
    ):
        size = get_size_gb(target_path)

        print(
            f"   SKIPPED - Already downloaded ({size:.2f} GiB)"
        )
        return "SKIPPED"

    # Recover a completed file from staging
    if (
        os.path.isfile(staged_path)
        and os.path.getsize(staged_path) > 1024 * 1024
    ):
        shutil.move(staged_path, target_path)

        print(
            f"   RECOVERED - {get_size_gb(target_path):.2f} GiB"
        )
        return "RECOVERED"

    print("   Downloading:", name)

    try:

        downloaded_path = hf_hub_download(
            repo_id=repo,
            filename=filename,
            token=HF_TOKEN,
            local_dir=ROOT,
            force_download=False
        )

        downloaded_path = str(downloaded_path)

        if not os.path.isfile(downloaded_path):
            raise FileNotFoundError(
                f"Downloaded file not found: {downloaded_path}"
            )

        if os.path.getsize(downloaded_path) < 1024 * 1024:
            raise RuntimeError("Downloaded file is unexpectedly small.")

        # Move to ComfyUI model directory
        if os.path.abspath(downloaded_path) != os.path.abspath(target_path):
            shutil.move(downloaded_path, target_path)

        size = get_size_gb(target_path)

        print(
            f"   COMPLETED - {size:.2f} GiB"
        )

        return "COMPLETED"

    except Exception as error:

        print("   FAILED:", name)
        print("   ERROR:", str(error)[:1200])

        if required:
            raise RuntimeError(
                f"Required model download failed: {name}"
            ) from error

        print("   OPTIONAL MODEL - Skipping and continuing.")
        return "OPTIONAL_FAILED"


# ------------------------------------------------------------
# 7. DOWNLOAD ALL MODELS
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("STARTING BFS LTX-2.5 MODEL DOWNLOADS")
print("=" * 65)

results = []

for index, item in enumerate(
    tqdm(
        MODEL_FILES,
        desc="Overall model progress",
        unit="model",
        dynamic_ncols=True
    ),
    start=1
):

    repo, filename, folder, required = item

    print(
        f"\n[{index}/{len(MODEL_FILES)}] {os.path.basename(filename)}"
    )

    result = download_one(item)

    results.append({
        "model": os.path.basename(filename),
        "status": result,
        "required": required
    })

# ------------------------------------------------------------
# 8. FINAL VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL MODEL VERIFICATION")
print("=" * 65)

for item in MODEL_FILES:

    repo, filename, folder, required = item

    name = os.path.basename(filename)

    target_path = os.path.join(
        MODEL_ROOT,
        folder,
        name
    )

    if os.path.isfile(target_path):
        print(
            f"[OK] {name} | {get_size_gb(target_path):.2f} GiB"
        )
    else:
        status = "MISSING REQUIRED" if required else "OPTIONAL MISSING"
        print(f"[{status}] {name}")

# ------------------------------------------------------------
# 9. SUMMARY
# ------------------------------------------------------------

completed = sum(
    1 for r in results
    if r["status"] in ["COMPLETED", "SKIPPED", "RECOVERED"]
)

optional_failed = sum(
    1 for r in results
    if r["status"] == "OPTIONAL_FAILED"
)

print("\n" + "=" * 65)
print("DOWNLOAD SUMMARY")
print("=" * 65)

print("Models processed:", len(results))
print("Available / completed:", completed)
print("Optional models skipped:", optional_failed)

print("\nComfyUI model directory:")
print(MODEL_ROOT)

print("\nWorkflow:")
print(WORKFLOW_PATH)

# Disk information
disk = shutil.disk_usage(ROOT)

print("\nDisk information:")
print(f"Total:     {disk.total / (1024**3):.2f} GiB")
print(f"Used:      {disk.used / (1024**3):.2f} GiB")
print(f"Free:      {disk.free / (1024**3):.2f} GiB")

print("\nCELL 3 FINISHED.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF authentication: OK
Authenticated account: adityapanda2608112

Workflow already exists. Skipping.
Workflow: /kaggle/temp/BFS_LTX25/workflows/workflow_ltx2.5_v1.1_head_swap_drag_and_drop.json

STARTING BFS LTX-2.5 MODEL DOWNLOADS


Overall model progress:   0%|          | 0/9 [00:00<?, ?model/s]


[1/9] ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors
   SKIPPED - Already downloaded (20.03 GiB)

[2/9] gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors
   SKIPPED - Already downloaded (14.32 GiB)

[3/9] gemma4_e2b_it_bf16.safetensors
   SKIPPED - Already downloaded (9.57 GiB)

[4/9] ltx-2.5-video-vae-bf16.safetensors
   SKIPPED - Already downloaded (1.37 GiB)

[5/9] ltx-2.5-audio-vae-bf16.safetensors
   SKIPPED - Already downloaded (0.34 GiB)

[6/9] ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors
   SKIPPED - Already downloaded (0.93 GiB)

[7/9] head_swap_ltx25_r128_v1.1.safetensors
   SKIPPED - Already downloaded (2.30 GiB)

[8/9] ltx-2.5-22b-ic-lora-pixel-spatial-upscaler-x2-1.0.safetensors
   Downloading: ltx-2.5-22b-ic-lora-pixel-spatial-upscaler-x2-1.0.safetensors
   FAILED: ltx-2.5-22b-ic-lora-pixel-spatial-upscaler-x2-1.0.safetensors
   ERROR: 403 Client Error. (Request ID: Root=1-6af74599-0e8d6fb841ab8c517c91f170;84dd6913-baa8-4306-83a2-63

In [ ]:

# BFS HOTFIX — Fix missing ComfyUI main.py
import os
import sys
import shutil
import subprocess

ROOT = "/kaggle/temp/BFS_LTX25"
COMFY_DIR = os.path.join(ROOT, "ComfyUI")
CUSTOM_NODES = os.path.join(
    COMFY_DIR, "custom_nodes", "ComfyUI-BFSNodes"
)

os.makedirs(ROOT, exist_ok=True)

# 1. Install ComfyUI if main.py is missing
if not os.path.isfile(os.path.join(COMFY_DIR, "main.py")):
    print("Installing ComfyUI...")

    if os.path.exists(COMFY_DIR):
        shutil.rmtree(COMFY_DIR)

    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/comfyanonymous/ComfyUI.git",
        COMFY_DIR
    ], check=True)
else:
    print("ComfyUI already installed.")

# 2. Install BFS custom nodes if missing
if not os.path.isdir(os.path.join(CUSTOM_NODES, ".git")):
    os.makedirs(os.path.dirname(CUSTOM_NODES), exist_ok=True)

    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/alisson-anjos/ComfyUI-BFSNodes.git",
        CUSTOM_NODES
    ], check=True)
else:
    print("BFS Nodes already installed.")

# 3. Install ComfyUI requirements
subprocess.run([
    sys.executable, "-m", "pip",
    "install", "-q", "-r",
    os.path.join(COMFY_DIR, "requirements.txt")
], check=True)

# 4. Verify
print("\n========== VERIFICATION ==========")
print("ComfyUI main.py:",
      os.path.isfile(os.path.join(COMFY_DIR, "main.py")))
print("BFS Nodes:", os.path.isdir(CUSTOM_NODES))

assert os.path.isfile(os.path.join(COMFY_DIR, "main.py"))

print("\nSUCCESS: ComfyUI repaired!")


In [19]:
#@title 4. Start private ComfyUI API backend (no public ComfyUI UI)
import os, sys, time, subprocess, requests, signal, psutil, pathlib

# Stop an old backend from a previous execution.
for p in psutil.process_iter(["pid", "cmdline"]):
    try:
        cmd = " ".join(p.info.get("cmdline") or [])
        if p.pid != os.getpid() and "ComfyUI/main.py" in cmd and "--port 8188" in cmd:
            p.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

COMFY_LOG = open(f"{ROOT}/comfy_backend.log", "w", buffering=1)
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1" if torch.cuda.device_count() >= 2 else "0"
# --lowvram forces aggressive weight offload instead of requiring the whole 22B transformer in one T4.
# --reserve-vram leaves headroom for CUDA context / VAE allocations.
COMFY_PROC = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
     "--disable-auto-launch", "--lowvram", "--reserve-vram", "1.5",
     "--preview-method", "none"],
    cwd=COMFY_DIR, env=env, stdout=COMFY_LOG, stderr=subprocess.STDOUT
)
print("ComfyUI backend PID:", COMFY_PROC.pid)

ready = False
for i in range(240):
    if COMFY_PROC.poll() is not None:
        print(open(f"{ROOT}/comfy_backend.log", errors="ignore").read()[-12000:])
        raise RuntimeError("ComfyUI backend exited during startup.")
    try:
        r = requests.get(COMFY_URL + "/system_stats", timeout=2)
        if r.ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if not ready:
    print(open(f"{ROOT}/comfy_backend.log", errors="ignore").read()[-16000:])
    raise TimeoutError("ComfyUI did not become ready within 8 minutes.")

print("Backend ready:", COMFY_URL)
print("Loaded custom node / object-info count:", len(requests.get(COMFY_URL+"/object_info", timeout=60).json()))
print("Backend log:", f"{ROOT}/comfy_backend.log")


ComfyUI backend PID: 1927
Backend ready: http://127.0.0.1:8188
Loaded custom node / object-info count: 1152
Backend log: /kaggle/temp/BFS_LTX25/comfy_backend.log


In [24]:

#@title 5. Repair Missing BFS Workflow Nodes + Restart ComfyUI

import os
import sys
import json
import time
import shutil
import subprocess
import pathlib
import requests
import psutil

ROOT = "/kaggle/temp/BFS_LTX25"
COMFY_DIR = os.path.join(ROOT, "ComfyUI")
CUSTOM_NODES_DIR = os.path.join(COMFY_DIR, "custom_nodes")

COMFY_URL = "http://127.0.0.1:8188"
WORKFLOW_NAME = "workflow_ltx2.5_v1.1_head_swap_drag_and_drop.json"

WORKFLOW_FILE = os.path.join(ROOT, "workflows", WORKFLOW_NAME)
WORKFLOW_API = os.path.join(ROOT, "workflow_ltx2.5_v1.1_api.json")
BACKEND_LOG = os.path.join(ROOT, "comfy_backend.log")

os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

# --------------------------------------------------
# 1. INSTALL CUSTOM NODES
# --------------------------------------------------

NODE_PACKS = {
    "ComfyUI-KJNodes":
        "https://github.com/kijai/ComfyUI-KJNodes.git",

    "ComfyUI-VideoHelperSuite":
        "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",

    "comfyui-workflow-to-api-converter-endpoint":
        "https://github.com/SethRobinson/comfyui-workflow-to-api-converter-endpoint.git",
}

print("\n========== INSTALLING CUSTOM NODES ==========")

for folder, repo in NODE_PACKS.items():
    target = os.path.join(CUSTOM_NODES_DIR, folder)

    if not os.path.isdir(os.path.join(target, ".git")):
        print(f"Installing {folder}...")

        if os.path.exists(target):
            shutil.rmtree(target)

        subprocess.run(
            ["git", "clone", "--depth", "1", repo, target],
            check=True
        )
    else:
        print(f"[OK] {folder} already installed")

# --------------------------------------------------
# 2. INSTALL DEPENDENCIES
# --------------------------------------------------

print("\n========== INSTALLING DEPENDENCIES ==========")

for folder in NODE_PACKS:
    req_file = os.path.join(
        CUSTOM_NODES_DIR, folder, "requirements.txt"
    )

    if os.path.isfile(req_file):
        print("Installing:", folder)

        subprocess.run(
            [
                sys.executable, "-m", "pip",
                "install", "-q", "-r", req_file
            ],
            check=True
        )

# --------------------------------------------------
# 3. STOP OLD COMFYUI BACKEND
# --------------------------------------------------

print("\n========== STOPPING OLD BACKEND ==========")

for proc in psutil.process_iter(["pid", "cmdline", "cwd"]):
    try:
        cmd = proc.info.get("cmdline") or []
        cwd = proc.info.get("cwd") or ""

        is_comfy = (
            any("main.py" in str(x) for x in cmd)
            and (
                COMFY_DIR in cwd
                or any(COMFY_DIR in str(x) for x in cmd)
            )
        )

        if is_comfy and proc.pid != os.getpid():
            print("Stopping PID:", proc.pid)
            proc.terminate()

            try:
                proc.wait(timeout=15)
            except psutil.TimeoutExpired:
                proc.kill()

    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

time.sleep(3)

# --------------------------------------------------
# 4. START COMFYUI
# --------------------------------------------------

print("\n========== STARTING COMFYUI ==========")

backend_log = open(BACKEND_LOG, "a", buffering=1)

backend_process = subprocess.Popen(
    [
        sys.executable,
        "main.py",
        "--listen", "127.0.0.1",
        "--port", "8188",
        "--disable-auto-launch",
        "--lowvram",
        "--reserve-vram", "1.5",
        "--preview-method", "none",
    ],
    cwd=COMFY_DIR,
    stdout=backend_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Backend PID:", backend_process.pid)

# --------------------------------------------------
# 5. WAIT FOR BACKEND
# --------------------------------------------------

print("\nWaiting for ComfyUI...")

backend_ready = False

for attempt in range(120):
    try:
        response = requests.get(
            COMFY_URL + "/system_stats",
            timeout=5
        )

        if response.ok:
            backend_ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(2)

if not backend_ready:
    raise RuntimeError(
        "ComfyUI failed to start. Check: " + BACKEND_LOG
    )

print("[OK] ComfyUI backend ONLINE")

# --------------------------------------------------
# 6. VERIFY CONVERTER ENDPOINT
# --------------------------------------------------

print("\n========== VERIFYING WORKFLOW CONVERTER ==========")

converter_ready = False

for attempt in range(30):
    try:
        test = requests.post(
            COMFY_URL + "/workflow/convert",
            json={"nodes": [], "links": []},
            timeout=10
        )

        if test.status_code != 404:
            converter_ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(2)

if not converter_ready:
    with open(BACKEND_LOG, "r", encoding="utf-8", errors="ignore") as f:
        log_tail = f.read()[-8000:]

    raise RuntimeError(
        "Workflow converter endpoint did not load.\n"
        "Check custom node installation and backend log:\n"
        + log_tail
    )

print("[OK] Workflow converter endpoint loaded")

# --------------------------------------------------
# 7. VALIDATE BACKEND NODES
# --------------------------------------------------

print("\n========== NODE VALIDATION ==========")

object_info = requests.get(
    COMFY_URL + "/object_info",
    timeout=120
).json()

required_nodes = [
    "AddLabel",
    "ImageConcatMulti",
    "ImageResizeKJv2",
    "VHS_LoadVideo",
    "VHS_VideoCombine",
]

missing_nodes = []

for node in required_nodes:
    if node in object_info:
        print("[FOUND]", node)
    else:
        print("[MISSING]", node)
        missing_nodes.append(node)

print("[VIRTUAL] GetNode / SetNode will be resolved by converter")

if missing_nodes:
    raise RuntimeError(
        "Missing backend nodes: " + str(missing_nodes)
    )

# --------------------------------------------------
# 8. LOAD OFFICIAL BFS WORKFLOW
# --------------------------------------------------

print("\n========== LOADING BFS WORKFLOW ==========")

if not os.path.isfile(WORKFLOW_FILE):
    raise FileNotFoundError(WORKFLOW_FILE)

with open(WORKFLOW_FILE, "r", encoding="utf-8") as f:
    workflow_ui = json.load(f)

print("[OK] Workflow loaded")
print("Workflow nodes:", len(workflow_ui.get("nodes", [])))

# --------------------------------------------------
# 9. CONVERT WORKFLOW WITH GET/SET ROUTING
# --------------------------------------------------

print("\n========== WORKFLOW API CONVERSION ==========")

response = requests.post(
    COMFY_URL + "/workflow/convert",
    json=workflow_ui,
    timeout=240
)

if not response.ok:
    print("HTTP Status:", response.status_code)
    print("Response:", response.text[:8000])
    raise RuntimeError("Workflow conversion failed")

API_WORKFLOW = response.json()

if not isinstance(API_WORKFLOW, dict) or not API_WORKFLOW:
    raise RuntimeError("Converter returned invalid or empty API workflow")

invalid_nodes = [
    node_id
    for node_id, node in API_WORKFLOW.items()
    if not isinstance(node, dict)
    or "class_type" not in node
    or "inputs" not in node
]

if invalid_nodes:
    raise RuntimeError(
        "Invalid API nodes: " + str(invalid_nodes[:30])
    )

virtual_nodes = [
    (node_id, node.get("class_type"))
    for node_id, node in API_WORKFLOW.items()
    if node.get("class_type") in ("GetNode", "SetNode")
]

if virtual_nodes:
    raise RuntimeError(
        "GetNode/SetNode were not resolved: " + str(virtual_nodes)
    )

with open(WORKFLOW_API, "w", encoding="utf-8") as f:
    json.dump(API_WORKFLOW, f, indent=2, ensure_ascii=False)

print("[OK] API workflow converted")
print("Converted API nodes:", len(API_WORKFLOW))
print("Saved:", WORKFLOW_API)

# --------------------------------------------------
# 10. MODEL FILE CHECK
# --------------------------------------------------

print("\n========== MODEL DIRECTORY CHECK ==========")

MODELS_DIR = os.path.join(COMFY_DIR, "models")

model_files = (
    [p.name for p in pathlib.Path(MODELS_DIR).rglob("*") if p.is_file()]
    if os.path.isdir(MODELS_DIR)
    else []
)

print("Models directory:", MODELS_DIR)
print("Model files found:", len(model_files))

print("\n========== CELL 5 COMPLETED ==========")
print("Backend PID:", backend_process.pid)
print("Backend log:", BACKEND_LOG)
print("API workflow:", WORKFLOW_API)



========== INSTALLING CUSTOM NODES ==========
[OK] ComfyUI-KJNodes already installed
[OK] ComfyUI-VideoHelperSuite already installed
Installing comfyui-workflow-to-api-converter-endpoint...


Cloning into '/kaggle/temp/BFS_LTX25/ComfyUI/custom_nodes/comfyui-workflow-to-api-converter-endpoint'...



========== INSTALLING DEPENDENCIES ==========
Installing: ComfyUI-KJNodes
Installing: ComfyUI-VideoHelperSuite

========== STOPPING OLD BACKEND ==========
Stopping PID: 2083

========== STARTING COMFYUI ==========
Backend PID: 2152

Waiting for ComfyUI...
[OK] ComfyUI backend ONLINE

========== VERIFYING WORKFLOW CONVERTER ==========
[OK] Workflow converter endpoint loaded

========== NODE VALIDATION ==========
[FOUND] AddLabel
[FOUND] ImageConcatMulti
[FOUND] ImageResizeKJv2
[FOUND] VHS_LoadVideo
[FOUND] VHS_VideoCombine
[VIRTUAL] GetNode / SetNode will be resolved by converter

========== LOADING BFS WORKFLOW ==========
[OK] Workflow loaded
Workflow nodes: 164

========== WORKFLOW API CONVERSION ==========
[OK] API workflow converted
Converted API nodes: 60
Saved: /kaggle/temp/BFS_LTX25/workflow_ltx2.5_v1.1_api.json

========== MODEL DIRECTORY CHECK ==========
Models directory: /kaggle/temp/BFS_LTX25/ComfyUI/models
Model files found: 36

========== CELL 5 COMPLETED ==========
Backen

In [30]:
#@title 6. BFS Gradio Face Swap + OFFICIAL Gradio Public Link

import os, sys, json, time, uuid, socket, requests, gradio as gr

# --------------------------------------------------
# CLEAN PREVIOUS GRADIO INSTANCE
# --------------------------------------------------

_previous_demo = globals().get("demo")
if _previous_demo is not None:
    try:
        _previous_demo.close()
    except Exception:
        pass

_previous_tunnel = globals().get("TUNNEL_PROC")
if _previous_tunnel and _previous_tunnel.poll() is None:
    _previous_tunnel.terminate()

COMFY_URL = "http://127.0.0.1:8188"
API_WORKFLOW_TEMPLATE = json.loads(json.dumps(API_WORKFLOW))
COMFY_CLIENT_ID = str(uuid.uuid4())

def get_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("0.0.0.0", 0))
        return s.getsockname()[1]

GRADIO_PORT = get_free_port()
print("Selected Gradio port:", GRADIO_PORT)


# --------------------------------------------------
# COMFYUI UPLOAD
# --------------------------------------------------

def comfy_upload(path):
    if not path or not os.path.isfile(path):
        raise ValueError("Upload both reference image and guide video.")

    with open(path, "rb") as f:
        r = requests.post(
            COMFY_URL + "/upload/image",
            files={"image": (os.path.basename(path), f)},
            data={"type": "input", "overwrite": "true"},
            timeout=300
        )

    r.raise_for_status()
    return r.json()["name"]


# --------------------------------------------------
# DETECT WORKFLOW LOADERS
# --------------------------------------------------

def candidate_nodes(workflow):
    imgs, vids = [], []

    for nid, node in workflow.items():
        cls = node.get("class_type", "")
        ins = node.get("inputs", {})

        if cls in ("LoadImage", "Load Image"):
            imgs.append((nid, cls, list(ins.keys())))

        if cls == "VHS_LoadVideo":
            vids.append((nid, cls, list(ins.keys())))

    return imgs, vids


IMAGE_CANDIDATES, VIDEO_CANDIDATES = candidate_nodes(
    API_WORKFLOW_TEMPLATE
)

print("Image loader candidates:", IMAGE_CANDIDATES)
print("Video loader candidates:", VIDEO_CANDIDATES)


def set_loader_input(node, filename, kind):
    ins = node.setdefault("inputs", {})

    preferred = (
        ["image"] if kind == "image"
        else ["video", "file", "filename"]
    )

    for key in preferred:
        if key in ins and not isinstance(ins[key], list):
            ins[key] = filename
            return

    raise KeyError(
        f"{kind} input not found: {list(ins.keys())}"
    )


# --------------------------------------------------
# QUEUE WORKFLOW
# --------------------------------------------------

def queue_and_wait(prompt, progress=gr.Progress()):
    r = requests.post(
        COMFY_URL + "/prompt",
        json={
            "prompt": prompt,
            "client_id": COMFY_CLIENT_ID
        },
        timeout=120
    )

    if not r.ok:
        raise RuntimeError("ComfyUI error: " + r.text[:5000])

    prompt_id = r.json()["prompt_id"]

    for n in range(3600):
        hist = requests.get(
            COMFY_URL + f"/history/{prompt_id}",
            timeout=30
        ).json()

        if prompt_id in hist:
            entry = hist[prompt_id]
            status_data = entry.get("status", {})

            if status_data.get("completed"):
                return entry

            messages = status_data.get("messages", [])

            if (
                status_data.get("status_str") == "error"
                or any(
                    isinstance(m, list)
                    and m
                    and m[0] == "execution_error"
                    for m in messages
                )
            ):
                raise RuntimeError(
                    "BFS generation failed. Check comfy_backend.log."
                )

        if n % 5 == 0:
            progress(
                min(0.95, 0.08 + n / 3600 * 0.87),
                desc="BFS generating..."
            )

        time.sleep(1)

    raise TimeoutError("Generation timed out.")


# --------------------------------------------------
# FIND GENERATED VIDEO
# --------------------------------------------------

def find_output_video(history):
    for node_out in history.get("outputs", {}).values():
        for key in ("videos", "gifs", "files"):
            for item in node_out.get(key, []) or []:
                filename = item.get("filename")

                if not filename:
                    continue

                subfolder = item.get("subfolder", "")
                typ = item.get("type", "output")

                base = (
                    os.path.join(COMFY_DIR, "temp")
                    if typ == "temp"
                    else OUTPUT_DIR
                )

                path = os.path.join(base, subfolder, filename)

                if os.path.isfile(path):
                    return path

    found = []

    for root, _, files in os.walk(OUTPUT_DIR):
        for name in files:
            if name.lower().endswith(
                (".mp4", ".webm", ".mov", ".mkv")
            ):
                path = os.path.join(root, name)
                found.append((os.path.getmtime(path), path))

    return max(found)[1] if found else None


# --------------------------------------------------
# BFS FACE SWAP FUNCTION
# --------------------------------------------------

def run_bfs(
    reference_image,
    guide_video,
    strength,
    seed,
    progress=gr.Progress()
):
    if not reference_image or not guide_video:
        raise gr.Error(
            "Upload a reference image and guide video."
        )

    if len(IMAGE_CANDIDATES) != 1 or len(VIDEO_CANDIDATES) != 1:
        raise gr.Error(
            f"Loader detection issue. "
            f"Images={IMAGE_CANDIDATES}; Videos={VIDEO_CANDIDATES}"
        )

    img_id = IMAGE_CANDIDATES[0][0]
    vid_id = VIDEO_CANDIDATES[0][0]

    image_name = comfy_upload(reference_image)
    video_name = comfy_upload(guide_video)

    graph = json.loads(
        json.dumps(API_WORKFLOW_TEMPLATE)
    )

    set_loader_input(
        graph[img_id],
        image_name,
        "image"
    )

    set_loader_input(
        graph[vid_id],
        video_name,
        "video"
    )

    for node in graph.values():
        cls = node.get("class_type", "").lower()
        ins = node.get("inputs", {})

        if "lora" in cls:
            for key in ("strength_model", "strength"):
                if key in ins and not isinstance(ins[key], list):
                    ins[key] = float(strength)

        if "sampler" in cls:
            for key in ("seed", "noise_seed"):
                if key in ins and not isinstance(ins[key], list):
                    ins[key] = int(seed)

    history = queue_and_wait(graph, progress)
    output_path = find_output_video(history)

    if not output_path:
        raise gr.Error(
            "Generation finished but output video was not found."
        )

    return (
        output_path,
        "Completed: " + os.path.basename(output_path)
    )


# --------------------------------------------------
# GRADIO INTERFACE
# --------------------------------------------------

with gr.Blocks(
    title="BFS LTX-2.5 R128 Face Swap",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("# BFS LTX-2.5 · R128 V1.1")
    gr.Markdown(
        "Upload a reference face image and a guide video."
    )

    with gr.Row():
        ref_in = gr.Image(
            type="filepath",
            label="Reference face image"
        )

        vid_in = gr.Video(
            label="Guide video"
        )

    with gr.Row():
        strength = gr.Slider(
            0.5, 1.5,
            value=1.0,
            step=0.05,
            label="BFS LoRA strength"
        )

        seed = gr.Number(
            value=123456789,
            precision=0,
            label="Seed"
        )

    gr.Markdown(
        f"Image loader: `{IMAGE_CANDIDATES}`  \n"
        f"Video loader: `{VIDEO_CANDIDATES}`"
    )

    run_btn = gr.Button(
        "Run BFS Face Swap",
        variant="primary"
    )

    out_video = gr.Video(
        label="Generated video"
    )

    status = gr.Textbox(
        label="Status",
        interactive=False
    )

    run_btn.click(
        run_bfs,
        inputs=[
            ref_in,
            vid_in,
            strength,
            seed
        ],
        outputs=[
            out_video,
            status
        ]
    )


# --------------------------------------------------
# LAUNCH OFFICIAL GRADIO SHARE LINK
# --------------------------------------------------

print("\n========== STARTING OFFICIAL GRADIO SHARE ==========")

launch_result = demo.launch(
    server_name="0.0.0.0",
    server_port=GRADIO_PORT,
    share=True,
    share_server_address=None,
    show_error=True,
    prevent_thread_lock=True,
    quiet=False
)

if isinstance(launch_result, tuple) and len(launch_result) >= 3:
    app, local_url, share_url = launch_result[:3]
else:
    local_url = None
    share_url = None

if not share_url:
    raise RuntimeError(
        "Gradio started, but the official public share URL was not returned. "
        "Check the Gradio launch output above."
    )

print("\n" + "=" * 70)
print("OFFICIAL GRADIO PUBLIC LINK:")
print(share_url)
print("=" * 70)
print("Local URL:", local_url)
print("Gradio port:", GRADIO_PORT)
print("ComfyUI backend:", COMFY_URL)
print("Generation runs on Kaggle GPU; open the public link from your PC.")
print("Keep this Kaggle session running while using the link.")

Selected Gradio port: 58987
Image loader candidates: [('461', 'LoadImage', ['image'])]
Video loader candidates: [('460', 'VHS_LoadVideo', ['video', 'custom_width', 'custom_height', 'skip_first_frames', 'select_every_nth', 'format', 'force_rate', 'frame_load_cap'])]

========== STARTING OFFICIAL GRADIO SHARE ==========
* Running on local URL:  http://0.0.0.0:58987
* Running on public URL: https://40304bbcd2b24c89ae.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



OFFICIAL GRADIO PUBLIC LINK:
https://40304bbcd2b24c89ae.gradio.live
Local URL: http://localhost:58987/
Gradio port: 58987
ComfyUI backend: http://127.0.0.1:8188
Generation runs on Kaggle GPU; open the public link from your PC.
Keep this Kaggle session running while using the link.


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2146, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1664, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

## Troubleshooting / limits

- **Hugging Face 401/403:** accept the Lightricks LTX-2.5 access terms and set a valid Kaggle Secret named `HF_TOKEN`.
- **Missing custom node:** inspect `BFS_LTX25/comfy_backend.log`; confirm `ComfyUI-BFSNodes` cloned and imported successfully.
- **Workflow converter error:** `comfy-ui2api` must match the workflow format and the node definitions returned by this exact ComfyUI + BFSNodes installation. Do not replace the graph with a generic LTX text-to-video graph.
- **Loader IDs ambiguous:** use the candidate node IDs printed in Cell 5/6 in the Advanced section. The notebook intentionally does not guess when more than one image/video loader exists.
- **Video upload rejected:** some custom video loaders use a dedicated upload route or a different filename widget. Check the loader's `class_type` and input names printed in Cell 5; adjust `comfy_upload()` / `set_loader_input()` for that node.
- **CUDA OOM:** reduce guide video duration/resolution in the official BFS graph or use the workflow's supported low-VRAM options. Do not quantize the BFS LoRA. The transformer is already INT8 ConvRot.
- **No output MP4:** inspect `BFS_LTX25/comfy_backend.log` and the ComfyUI history response. Some workflows save video under a node-specific output key; extend `find_output_video()` if the workflow uses a custom output node.

This notebook is a **headless ComfyUI inference backend with a Gradio-only public interface**. It does not expose or ask you to use the ComfyUI browser UI. It is not a direct Wan2GP port: BFS's custom guide/identity conditioning is preserved by running the creator's actual workflow and BFSNodes implementation. Kaggle hardware, package versions and workflow revisions can still cause runtime-specific fixes; the notebook performs startup checks rather than claiming an untested run is guaranteed.